# C-MAPSS model experiment runner

In [1]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
current_directory = Path.cwd().resolve()
PROJECT_ROOT = next((directory for directory in (current_directory, *current_directory.parents)if (directory / 'pyproject.toml').is_file()), None)

if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate the project root')

CODE_ROOT = PROJECT_ROOT / 'code'

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

os.chdir(PROJECT_ROOT)
os.environ.setdefault('MLFLOW_ALLOW_FILE_STORE', 'true')
os.environ.setdefault('MLFLOW_TRACKING_URI', f'file:{PROJECT_ROOT / "mlruns"}')
os.environ.setdefault('CMAPSS_MLFLOW_EXPERIMENT', 'cmapss-preprocessing')
os.environ.setdefault('CMAPSS_MLFLOW_TRAINING_EXPERIMENT', 'cmapss-training')

print(f'Project root: {PROJECT_ROOT}')

Project root: /home/aanchal/nasa_c_mapss


In [3]:
from src.training.lstm import train_lstm
from src.build_spark import spark_session_context
from src.training.tree_models import train_random_forest
from src.data_processing.preprocessing import run_subset_preprocessing
from src.training.tree_models.run_tabular_training import run_training
from src.training.evaluation.test_evaluation import evaluate_test_data

from src.data_processing import build_endpoint_sequences
from src.tracking import load_training_feature_columns, load_training_model
from src.training.evaluation.engine_endpoint_evaluation import (evaluate_prediction_diagnostics,
                                                                prepare_pseudo_test_validation,
                                                                select_pseudo_test_endpoints)
from src.training.evaluation import evaluate_predictions, life_ratio_to_rul

In [4]:
SUBSETS = ('FD001', 'FD002', 'FD003', 'FD004')
RAW_DATA_DIR = PROJECT_ROOT / 'Data' / 'CMAPSSData'
PROCESSED_DATA_DIR = PROJECT_ROOT / 'Data' / 'processed'

RUN_TESTS = True
RUN_PREPROCESSING = False
RUN_TRAINING = True

In [5]:
# When RUN_PREPROCESSING is False, populate these with runs produced by the life-ratio preprocessing pipeline.
BASELINE_PREPROCESSING_RUN_IDS = {
"FD001":	"ef5500dcac8046f7887ebdd522a99337",
"FD002":	"72c8b532ad544a1fa51ba2129cd409f8",
"FD003":	"717209bc10d94bdf9647e16cf238f874",
"FD004":	"30c643cf3efd4db08801af7572f537da",
}
TEMPORAL_PREPROCESSING_RUN_IDS = {
"FD001":	"80aaff25e00d454884f456b832edbf5d",
"FD002":	"8c8a1bdacfd34bd18a125704ea05b86e",
"FD003":	"bfe3aff0f4af4fbda6f36ed04d229dde",
"FD004":	"cf4b0ae412bb4dc289a386ddf1d1fb59",
}

preprocessing_run_ids = {
    **{(subset, 'baseline'): run_id for subset, run_id in BASELINE_PREPROCESSING_RUN_IDS.items()},
    **{(subset, 'temporal'): run_id for subset, run_id in TEMPORAL_PREPROCESSING_RUN_IDS.items()},
}
preprocessing_results = []

In [6]:
if RUN_PREPROCESSING:
    for feature_set, include_temporal_features in (
        ('baseline', False),
        ('temporal', True),
    ):
        for subset in SUBSETS:
            print(f'\n[{subset}] preprocessing {feature_set} features...', flush=True)

            with spark_session_context(app_name=f'cmapss-{subset}-{feature_set}-notebook') as spark:
                result = run_subset_preprocessing(
                    spark=spark,
                    subset=subset,
                    raw_data_dir=RAW_DATA_DIR,
                    output_dir=PROCESSED_DATA_DIR,
                    include_temporal_features=include_temporal_features,
                )

            preprocessing_run_ids[(subset, feature_set)] = result.run_id
            preprocessing_results.append({
                'subset': subset,
                'feature_set': feature_set,
                'preprocessing_run_id': result.run_id,
                'source': 'created',
                'feature_count': result.feature_count,
                'train_rows': result.train_row_count,
            })
else:
    preprocessing_results.extend(
        {
            'subset': subset,
            'feature_set': feature_set,
            'preprocessing_run_id': run_id,
            'source': 'existing',
        }
        for (subset, feature_set), run_id in preprocessing_run_ids.items()
    )

In [7]:
required_preprocessing_runs = {
    (subset, feature_set)
    for subset in SUBSETS
    for feature_set in ('baseline', 'temporal')
}
missing_runs = required_preprocessing_runs - preprocessing_run_ids.keys()
if missing_runs:
    raise RuntimeError(
        'Missing life-ratio preprocessing runs. Enable RUN_PREPROCESSING or '
        f'provide run IDs for: {sorted(missing_runs)}'
    )

pd.DataFrame(preprocessing_results)

,subset,feature_set,preprocessing_run_id,source
0,FD001,baseline,ef5500dcac8046f7887ebdd522a99337,existing
1,FD002,baseline,72c8b532ad544a1fa51ba2129cd409f8,existing
2,FD003,baseline,717209bc10d94bdf9647e16cf238f874,existing
3,FD004,baseline,30c643cf3efd4db08801af7572f537da,existing
4,FD001,temporal,80aaff25e00d454884f456b832edbf5d,existing
5,FD002,temporal,8c8a1bdacfd34bd18a125704ea05b86e,existing
6,FD003,temporal,bfe3aff0f4af4fbda6f36ed04d229dde,existing
7,FD004,temporal,cf4b0ae412bb4dc289a386ddf1d1fb59,existing


## Random Forest training


In [ ]:
training_results = []
random_forest_results = []

if RUN_TRAINING:
    for feature_set in ('baseline', 'temporal'):
        for subset in SUBSETS:
            preprocessing_run_id = preprocessing_run_ids[(subset, feature_set)]
            print(f'\n[{subset}] training Random Forest with {feature_set} features...', flush=True)

            training_run_id = train_random_forest(
                subset_id=subset,
                preprocessing_run_id=preprocessing_run_id,
                processed_data_dir=PROCESSED_DATA_DIR,
            )
            test_metrics = evaluate_test_data(
                subset_id=subset,
                training_run_id=training_run_id,
                processed_data_dir=PROCESSED_DATA_DIR,
                raw_data_dir=RAW_DATA_DIR,
            )
            random_forest_results.append({
                'subset': subset,
                'model': 'random_forest',
                'feature_set': feature_set,
                'preprocessing_run_id': preprocessing_run_id,
                'training_run_id': training_run_id,
                **{f'test_{name}': value for name, value in test_metrics.items()},
            })

training_results.extend(random_forest_results)
pd.DataFrame(random_forest_results)


[FD001] training Random Forest with baseline features...
[FD001] fitting random-forest on 16,260 rows and 15baseline features with target=life_ratio...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:    3.2s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    8.7s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished
2026/08/25 02:07:59 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /home/aanchal/nasa_c_mapss
2026/08/25 02:08:12 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.



[FD002] training Random Forest with baseline features...


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished


[FD002] fitting random-forest on 43,289 rows and 21baseline features with target=life_ratio...


...

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:  2.0min
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  4.5min finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished
2026/08/25 02:28:36 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /home/aanchal/nasa_c_mapss
2026/08/25 02:28:40 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.



[FD004] training Random Forest with temporal features...


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished


[FD004] fitting random-forest on 48,733 rows and 315temporal features with target=life_ratio...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.


## XGBoost training


In [8]:
xgboost_results = []

if RUN_TRAINING:
    for feature_set in ('baseline', 'temporal'):
        for subset in SUBSETS:
            preprocessing_run_id = preprocessing_run_ids[(subset, feature_set)]
            print(f'\n[{subset}] training XGBoost with {feature_set} features...', flush=True)

            result = run_training(
                model_type='xgboost',
                subset_id=subset,
                preprocessing_run_id=preprocessing_run_id,
                processed_data_dir=PROCESSED_DATA_DIR,
                raw_data_dir=RAW_DATA_DIR,
            )
            xgboost_results.append({
                'subset': subset,
                'model': 'xgboost',
                'feature_set': feature_set,
                'preprocessing_run_id': preprocessing_run_id,
                'training_run_id': result['training_run_id'],
                **{f'test_{name}': value for name, value in result['test_metrics'].items()},
            })

training_results.extend(xgboost_results)
pd.DataFrame(xgboost_results)


[FD001] training XGBoost with baseline features...
[FD001] starting model training...
[FD001] fitting xgboost on 16,260 rows and 15baseline features with target=life_ratio...


2026/08/25 03:15:04 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /home/aanchal/nasa_c_mapss
2026/08/25 03:15:05 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /home/aanchal/nasa_c_mapss
2026/08/25 03:15:05 INFO mlflow.utils.environment: Detected uv project at /home/aanchal/nasa_c_mapss. Attempting to export requirements via 'uv export'.
2026/08/25 03:15:05 INFO mlflow.utils.uv_utils: Exported 94 dependencies via uv
2026/08/25 03:15:05 INFO mlflow.utils.environment: Successfully exported 94 requirements from uv project. Skipping package capture based inference.
2026/08/25 03:15:05 INFO mlflow.utils.uv_utils: Extracted 1 private index URL(s) from uv.lock
2026/08/25 03:15:05 WARNING mlflow.utils.uv_utils: Private package indexes detected in uv lockfile. Ensure credentials are available at model load time via UV_INDEX_* environment variables or .netrc file.
2026/08/25 03:15:06 WARNING mlflow.utils.environment: Fa

[FD001] evaluating the test set...

[FD002] training XGBoost with baseline features...
[FD002] starting model training...
[FD002] fitting xgboost on 43,289 rows and 21baseline features with target=life_ratio...


2026/08/25 03:15:13 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /home/aanchal/nasa_c_mapss
2026/08/25 03:15:14 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /home/aanchal/nasa_c_mapss
2026/08/25 03:15:14 INFO mlflow.utils.environment: Detected uv project at /home/aanchal/nasa_c_mapss. Attempting to export requirements via 'uv export'.
2026/08/25 03:15:14 INFO mlflow.utils.uv_utils: Exported 94 dependencies via uv
2026/08/25 03:15:14 INFO mlflow.utils.environment: Successfully exported 94 requirements from uv project. Skipping package capture based inference.
2026/08/25 03:15:14 INFO mlflow.utils.uv_utils: Extracted 1 private index URL(s) from uv.lock
2026/08/25 03:15:14 WARNING mlflow.utils.uv_utils: Private package indexes detected in uv lockfile. Ensure credentials are available at model load time via UV_INDEX_* environment variables or .netrc file.
2026/08/25 03:15:14 WARNING mlflow.utils.environment: Fa

[FD002] evaluating the test set...

[FD003] training XGBoost with baseline features...
[FD003] starting model training...
[FD003] fitting xgboost on 19,979 rows and 16baseline features with target=life_ratio...


2026/08/25 03:15:17 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /home/aanchal/nasa_c_mapss
2026/08/25 03:15:17 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /home/aanchal/nasa_c_mapss
2026/08/25 03:15:17 INFO mlflow.utils.environment: Detected uv project at /home/aanchal/nasa_c_mapss. Attempting to export requirements via 'uv export'.
2026/08/25 03:15:17 INFO mlflow.utils.uv_utils: Exported 94 dependencies via uv
2026/08/25 03:15:17 INFO mlflow.utils.environment: Successfully exported 94 requirements from uv project. Skipping package capture based inference.
2026/08/25 03:15:17 INFO mlflow.utils.uv_utils: Extracted 1 private index URL(s) from uv.lock
2026/08/25 03:15:17 WARNING mlflow.utils.uv_utils: Private package indexes detected in uv lockfile. Ensure credentials are available at model load time via UV_INDEX_* environment variables or .netrc file.
2026/08/25 03:15:18 WARNING mlflow.utils.environment: Fa

[FD003] evaluating the test set...

[FD004] training XGBoost with baseline features...
[FD004] starting model training...
[FD004] fitting xgboost on 48,733 rows and 21baseline features with target=life_ratio...


2026/08/25 03:15:24 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /home/aanchal/nasa_c_mapss
2026/08/25 03:15:25 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /home/aanchal/nasa_c_mapss
2026/08/25 03:15:25 INFO mlflow.utils.environment: Detected uv project at /home/aanchal/nasa_c_mapss. Attempting to export requirements via 'uv export'.
2026/08/25 03:15:25 INFO mlflow.utils.uv_utils: Exported 94 dependencies via uv
2026/08/25 03:15:25 INFO mlflow.utils.environment: Successfully exported 94 requirements from uv project. Skipping package capture based inference.
2026/08/25 03:15:25 INFO mlflow.utils.uv_utils: Extracted 1 private index URL(s) from uv.lock
2026/08/25 03:15:25 WARNING mlflow.utils.uv_utils: Private package indexes detected in uv lockfile. Ensure credentials are available at model load time via UV_INDEX_* environment variables or .netrc file.
2026/08/25 03:15:25 WARNING mlflow.utils.environment: Fa

[FD004] evaluating the test set...

[FD001] training XGBoost with temporal features...
[FD001] starting model training...
[FD001] fitting xgboost on 16,260 rows and 225temporal features with target=life_ratio...


OverflowError: math range error

## LightGBM training


In [ ]:
lightgbm_results = []

if RUN_TRAINING:
    for feature_set in ('baseline', 'temporal'):
        for subset in SUBSETS:
            preprocessing_run_id = preprocessing_run_ids[(subset, feature_set)]
            print(f'\n[{subset}] training LightGBM with {feature_set} features...', flush=True)

            result = run_training(model_type='lightgbm', subset_id=subset,
                                  preprocessing_run_id=preprocessing_run_id,
                                  processed_data_dir=PROCESSED_DATA_DIR, raw_data_dir=RAW_DATA_DIR)

            lightgbm_results.append({
                    'subset': subset,
                    'model': 'lightgbm',
                    'feature_set': feature_set,
                    'preprocessing_run_id': preprocessing_run_id,
                    'training_run_id': result['training_run_id'],
                    **{f'test_{name}': value for name, value in result['test_metrics'].items()}})

training_results.extend(lightgbm_results)
pd.DataFrame(lightgbm_results)

## LSTM training


In [ ]:
lstm_results = []

if RUN_TRAINING:
    for feature_set, sequence_feature_set in (
        ('baseline', 'base_sequence'),
        ('temporal', 'temporal_sequence'),
    ):
        for subset in SUBSETS:
            preprocessing_run_id = preprocessing_run_ids[(subset, feature_set)]
            print(f'\n[{subset}] training LSTM with {feature_set} features...', flush=True)

            result = train_lstm(
                subset_id=subset,
                preprocessing_run_id=preprocessing_run_id,
                processed_data_dir=PROCESSED_DATA_DIR,
                raw_data_dir=RAW_DATA_DIR,
                feature_set=sequence_feature_set,
            )
            lstm_results.append({
                'subset': subset,
                'model': 'lstm',
                'feature_set': sequence_feature_set,
                'preprocessing_run_id': preprocessing_run_id,
                'training_run_id': result['training_run_id'],
                **{f'test_{name}': value for name, value in result['test_metrics'].items()},
            })

training_results.extend(lstm_results)
pd.DataFrame(lstm_results)

## Model comparison


In [ ]:
results = pd.DataFrame(training_results)

comparison_columns = [
    'test_mae',
    'test_rmse',
    'test_nasa_score',
    'test_bias',
    'test_late_prediction_rate',
    'test_worst_positive_error',
    'test_worst_negative_error',
    'test_largest_nasa_contribution',
    'test_top_3_nasa_contribution',
]

results.set_index(['subset', 'model', 'feature_set'])[comparison_columns].round(3)

## Robustness validation


In [ ]:
ROBUSTNESS_SEEDS = tuple(range(42, 52))

ROBUSTNESS_CANDIDATES = (
    {'subset': 'FD001', 'candidate': 'temporal_lstm', 'role': 'champion', 'model_family': 'lstm', 'preprocessing_run_id': TEMPORAL_PREPROCESSING_RUN_IDS['FD001'], 'training_run_id': 'e9ee327ea7814c0bafc8179e188fcba7'},
    {'subset': 'FD001', 'candidate': 'temporal_xgboost', 'role': 'competitor', 'model_family': 'tabular', 'preprocessing_run_id': TEMPORAL_PREPROCESSING_RUN_IDS['FD001'], 'training_run_id': 'd37624ea98f5459db673a69c5dc56951'},
    {'subset': 'FD002', 'candidate': 'temporal_lstm', 'role': 'champion', 'model_family': 'lstm', 'preprocessing_run_id': TEMPORAL_PREPROCESSING_RUN_IDS['FD002'], 'training_run_id': '470f26004ed0462abc492a62872ce600'},
    {'subset': 'FD002', 'candidate': 'temporal_xgboost', 'role': 'competitor', 'model_family': 'tabular', 'preprocessing_run_id': TEMPORAL_PREPROCESSING_RUN_IDS['FD002'], 'training_run_id': '1b7584f00aad4746a59ff0103f2448c3'},
    {'subset': 'FD003', 'candidate': 'base_lstm', 'role': 'champion', 'model_family': 'lstm', 'preprocessing_run_id': BASELINE_PREPROCESSING_RUN_IDS['FD003'], 'training_run_id': '1ad1630ccad347b99abc71ed1317566b'},
    {'subset': 'FD003', 'candidate': 'temporal_lstm', 'role': 'competitor', 'model_family': 'lstm', 'preprocessing_run_id': TEMPORAL_PREPROCESSING_RUN_IDS['FD003'], 'training_run_id': '9a1fce6e99cd401a9891a8ca210c629a'},
    {'subset': 'FD004', 'candidate': 'temporal_xgboost', 'role': 'champion', 'model_family': 'tabular', 'preprocessing_run_id': TEMPORAL_PREPROCESSING_RUN_IDS['FD004'], 'training_run_id': '1b90ed34b6964e58bcef186c090cfc52'},
    {'subset': 'FD004', 'candidate': 'temporal_random_forest', 'role': 'competitor', 'model_family': 'tabular', 'preprocessing_run_id': TEMPORAL_PREPROCESSING_RUN_IDS['FD004'], 'training_run_id': '9765b1d0d6e34ec49360fa8b9bbeb09f'},
)

In [ ]:
robustness_results = []

for candidate in ROBUSTNESS_CANDIDATES:
    validation_path = (PROCESSED_DATA_DIR / candidate['subset'] / candidate['preprocessing_run_id'] / 'validation')
    validation_dataframe = pd.read_parquet(validation_path)
    
    feature_columns = load_training_feature_columns(candidate['training_run_id'])
    validation_dataframe[feature_columns] = (validation_dataframe[feature_columns].fillna(0).astype(np.float32))
    
    model = load_training_model(candidate['training_run_id'])

    for seed in ROBUSTNESS_SEEDS:
        if candidate['model_family'] == 'lstm':
            metadata = select_pseudo_test_endpoints(validation_dataframe, seed=seed)
            sequences = build_endpoint_sequences(validation_dataframe, metadata, feature_columns, sequence_length=30)
            
            predictions = np.asarray([np.asarray(model.predict(sequence[np.newaxis, ...])).reshape(-1)[0] for sequence in sequences])
            targets = metadata['life_ratio']
            
        else:
            features, targets, metadata = prepare_pseudo_test_validation(validation_dataframe, feature_columns, seed=seed)
            predictions = model.predict(features)

        target_rul = life_ratio_to_rul(metadata['cycle'], targets)
        predicted_rul = life_ratio_to_rul(metadata['cycle'], predictions)
        metrics = evaluate_predictions(target_rul, predicted_rul)
        _, tail_metrics = evaluate_prediction_diagnostics(metadata, predictions)
        robustness_results.append({**candidate, 'seed': seed, **metrics, **tail_metrics})

robustness_by_seed = pd.DataFrame(robustness_results)
robustness_summary = (robustness_by_seed.groupby(['subset', 'candidate', 'role', 'preprocessing_run_id', 'training_run_id'], as_index=False)
                      .agg(mean_mae=('mae', 'mean'), 
                      median_mae=('mae', 'median'), 
                      mean_rmse=('rmse', 'mean'), 
                      median_rmse=('rmse', 'median'), 
                      mean_nasa_score=('nasa_score', 'mean'), 
                      median_nasa_score=('nasa_score', 'median'), 
                      worst_seed_nasa_score=('nasa_score', 'max'), 
                      worst_positive_error=('worst_positive_error', 'max')))

champion_scores = robustness_by_seed.loc[robustness_by_seed['role'] == 'champion', ['subset', 'seed', 'nasa_score']
                                         ].rename(columns={'nasa_score': 'champion_nasa_score'})

competitor_scores = robustness_by_seed.loc[robustness_by_seed['role'] == 'competitor', ['subset', 'seed', 'nasa_score']
                                           ].rename(columns={'nasa_score': 'competitor_nasa_score'})

win_rates = champion_scores.merge(competitor_scores, on=['subset', 'seed'])
win_rates = (win_rates.assign(champion_win=lambda frame: frame['champion_nasa_score'] < frame['competitor_nasa_score'])
             .groupby('subset', as_index=False)['champion_win']
             .mean()
             .rename(columns={'champion_win': 'champion_nasa_win_rate'}))

robustness_summary.merge(win_rates, on='subset', how='left').round(3)

## Feature-importance analysis


In [ ]:
import mlflow
import mlflow.lightgbm as mlflow_lightgbm
import mlflow.sklearn as mlflow_sklearn
import mlflow.xgboost as mlflow_xgboost
from src.tracking import configure_mlflow

feature_importance_results = []
model_names = {'RandomForestRegressor': 'random_forest',
               'XGBRegressor': 'xgboost',
               'LGBMRegressor': 'lightgbm'}

model_loaders = {'random_forest': mlflow_sklearn.load_model,
                 'xgboost': mlflow_xgboost.load_model,
                 'lightgbm': mlflow_lightgbm.load_model}

configure_mlflow()

tracked_runs = mlflow.search_runs(
        experiment_names=[os.getenv('CMAPSS_MLFLOW_TRAINING_EXPERIMENT', 'cmapss-training')],
        output_format='pandas')

latest_model_runs = (tracked_runs[
        tracked_runs['status'].eq('FINISHED')
        & tracked_runs['params.model_type'].isin(model_names)
        & tracked_runs['tags.feature_set'].isin(('baseline', 'temporal'))]
        .sort_values('start_time', ascending=False)
        .drop_duplicates(['params.subset', 'params.model_type', 'tags.feature_set']))


for result in latest_model_runs.to_dict('records'):
        
    result['model'] = model_names[result['params.model_type']]
    result['training_run_id'] = result['run_id']
    result['subset'] = result['params.subset']
    result['feature_set'] = result['tags.feature_set']
    
    model = model_loaders[result['model']](f"runs:/{result['training_run_id']}/model")
    
    feature_columns = load_training_feature_columns(result['training_run_id'])

    feature_importance_results.extend(
            {'subset': result['subset'], 'model': result['model'], 'feature_set': result['feature_set'],
             'training_run_id': result['training_run_id'],
             'feature': feature, 'importance': importance}            
            for feature, importance in zip(feature_columns, model.feature_importances_))

feature_importance_results = pd.DataFrame(feature_importance_results)

feature_importance_results = (feature_importance_results
        .sort_values(['subset', 'model', 'feature_set', 'importance'], ascending=[True, True, True, False])
        .groupby(['subset', 'model', 'feature_set'], as_index=False).head(20)
        .reset_index(drop=True))

In [ ]:
feature_importance_results[(feature_importance_results.model=="xgboost") 
                           & (feature_importance_results.subset=="FD001")
                           & (feature_importance_results.feature_set=="temporal")].head(10)

In [ ]:
feature_importance_results[(feature_importance_results.model=="xgboost") 
                           & (feature_importance_results.subset=="FD002")
                           & (feature_importance_results.feature_set=="temporal")].head(10)

In [ ]:
feature_importance_results[(feature_importance_results.model=="xgboost") 
                           & (feature_importance_results.subset=="FD003")
                           & (feature_importance_results.feature_set=="temporal")].head(10)

In [ ]:
feature_importance_results[(feature_importance_results.model=="xgboost") 
                           & (feature_importance_results.subset=="FD004")
                           & (feature_importance_results.feature_set=="temporal")].head(10)